In [2]:
import pandas as pd
from pymongo import MongoClient

# temizlenmis csv yi okuyorum
df = pd.read_csv('../data/processed/cleaned_tweets.csv')

# mongodb ye baglanma
try:
    client = MongoClient('mongodb://localhost:27017/')
    print("MongoDB baglantisi basarili!")

    # veritabani ve koleksiyon
    db = client['twitter_db_pipeline']
    collection = db['tweetler']

    # eger onceden veri varsa temizleyelim ki cift kayit olmasin
    collection.delete_many({})

    # pandas dataframe i dict listesine cevirip toplu yukleme (bulk insert)
    records = df.to_dict('records')
    collection.insert_many(records)
    print(f"{len(records)} dokuman MongoDB'ye yuklendi.")

except Exception as e:
    print(f"Hata: {e}")

MongoDB baglantisi basarili!
14604 dokuman MongoDB'ye yuklendi.


**MongoDB Sema Tasarimi:**
Verileri "Flat Document" (duz dokuman) seklinde tuttum. Her tweet bir JSON/BSON dokumani olarak kaydedildi. Tweet metni, havayolu, duygu durumu, tarih gibi hersey tek bir dokumanda. Tablolar arasi JOIN islemine gerek olmadigindan bu yapida okuma hizi cok iyi oluyor.

In [3]:
print("--- 1. find() Sorgulari ---")

# sorgu 1: $and ile coklu kosul - hem pozitif hem guven skoru 1.0 olan virgin america tweetleri
# sadece text ve confidence geri donecek (projeksiyon)
print("\nSorgu 1: Virgin America - pozitif VE guven=1.0 (coklu kosul + projeksiyon):")
sonuc1 = collection.find(
    {"$and": [
        {"airline": "Virgin America"},
        {"airline_sentiment": "positive"},
        {"airline_sentiment_confidence": 1.0}
    ]},
    {"text": 1, "airline_sentiment_confidence": 1, "_id": 0}
).sort("retweet_count", -1).limit(3)
for doc in sonuc1:
    print(doc)

# sorgu 2: $or kullanimi - ya rotar ya iptal yasayan ve guven skoru 0.8 ustu olanlar
print("\nSorgu 2: Rotar VEYA iptal + guven > 0.8 ($or + $gt):")
sonuc2 = collection.find(
    {
        "$or": [
            {"negativereason": "Late Flight"},
            {"negativereason": "Cancelled Flight"}
        ],
        "airline_sentiment_confidence": {"$gt": 0.8}
    },
    {"airline": 1, "negativereason": 1, "airline_sentiment_confidence": 1, "_id": 0}
).limit(3)
for doc in sonuc2:
    print(doc)

# sorgu 3: $ne ve $exists birlesimi - konum bilgisi "Bilinmiyor" olmayan,
# yani gercekten konum paylasan kullanicilarin tweetleri + skip ile sayfalama
print("\nSorgu 3: Konum paylasan kullanicilar ($ne + skip/limit sayfalama):")
sonuc3 = collection.find(
    {
        "tweet_location": {"$ne": "Bilinmiyor", "$exists": True},
        "airline_sentiment": "negative"
    },
    {"name": 1, "tweet_location": 1, "airline": 1, "_id": 0}
).skip(10).limit(3)
for doc in sonuc3:
    print(doc)

--- 1. find() Sorgulari ---

Sorgu 1: Virgin America - pozitif VE guven=1.0 (coklu kosul + projeksiyon):
{'airline_sentiment_confidence': 1.0, 'text': '@VirginAmerica Flying LAX to SFO and after looking at the awesome movie lineup I actually wish I was on a long haul.'}
{'airline_sentiment_confidence': 1.0, 'text': "Always have it together!!! You're welcome! RT @VirginAmerica: @jessicajaymes You're so welcome."}
{'airline_sentiment_confidence': 1.0, 'text': '@VirginAmerica thanks for gate checking my baggage on your full flight dfw-lax 883 and giving me early boarding too #sweet'}

Sorgu 2: Rotar VEYA iptal + guven > 0.8 ($or + $gt):
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Late Flight', 'airline': 'Virgin America'}
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Cancelled Flight', 'airline': 'Virgin America'}
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Late Flight', 'airline': 'Virgin America'}

Sorgu 3: Konum paylasan kullanicilar ($ne + skip/lim

In [4]:
print("--- 2. Aggregation Pipeline Sorgulari ---")

# sorgu 4: her havayolu icin ayni anda birden fazla istatistik hesapla
# $group icinde $sum, $avg, $min, $max bir arada
print("\nSorgu 4: Havayolu bazli detayli istatistikler (sum + avg + min + max):")
pipeline1 = [
    {"$group": {
        "_id": "$airline",
        "toplam_tweet": {"$sum": 1},
        "ort_guven": {"$avg": "$airline_sentiment_confidence"},
        "min_guven": {"$min": "$airline_sentiment_confidence"},
        "max_rt": {"$max": "$retweet_count"}
    }},
    {"$sort": {"toplam_tweet": -1}}
]
for doc in collection.aggregate(pipeline1):
    print(doc)

# sorgu 5: $match + $group + $project + $cond - negatif tweetleri al,
# havayoluna gore grupla, eger ortalama guven 0.9 ustuyse "yuksek" degilse "dusuk" etiketle
print("\nSorgu 5: Negatif tweetlerde havayolu bazli guven seviyesi etiketi ($cond):")
pipeline2 = [
    {"$match": {"airline_sentiment": "negative"}},
    {"$group": {
        "_id": "$airline",
        "negatif_sayisi": {"$sum": 1},
        "ort_guven": {"$avg": "$airline_sentiment_confidence"}
    }},
    {"$project": {
        "negatif_sayisi": 1,
        "ort_guven": {"$round": ["$ort_guven", 3]},
        "guven_seviyesi": {
            "$cond": {
                "if": {"$gte": ["$ort_guven", 0.9]},
                "then": "yuksek",
                "else": "dusuk"
            }
        }
    }},
    {"$sort": {"negatif_sayisi": -1}}
]
for doc in collection.aggregate(pipeline2):
    print(doc)

# sorgu 6: $addFields ile yeni alan ekleyerek pipeline icinde veri zenginlestirme
# sikayet sebeplerine gore grupla ve toplam icindeki yuzdelerini hesapla
print("\nSorgu 6: Sikayet sebeplerinin yuzdesel dagilimi ($addFields):")
toplam_negatif = collection.count_documents({"airline_sentiment": "negative"})
pipeline3 = [
    {"$match": {"airline_sentiment": "negative", "negativereason": {"$ne": "Belirtilmedi"}}},
    {"$group": {"_id": "$negativereason", "adet": {"$sum": 1}}},
    {"$addFields": {
        "yuzde": {
            "$round": [{"$multiply": [{"$divide": ["$adet", toplam_negatif]}, 100]}, 1]
        }
    }},
    {"$sort": {"adet": -1}},
    {"$limit": 5}
]
for doc in collection.aggregate(pipeline3):
    print(doc)

--- 2. Aggregation Pipeline Sorgulari ---

Sorgu 4: Havayolu bazli detayli istatistikler (sum + avg + min + max):
{'_id': 'United', 'toplam_tweet': 3822, 'ort_guven': 0.9008776818419676, 'min_guven': 0.335, 'max_rt': 7}
{'_id': 'US Airways', 'toplam_tweet': 2913, 'ort_guven': 0.9215784414692757, 'min_guven': 0.34, 'max_rt': 44}
{'_id': 'American', 'toplam_tweet': 2723, 'ort_guven': 0.9162592728608153, 'min_guven': 0.3367, 'max_rt': 5}
{'_id': 'Southwest', 'toplam_tweet': 2420, 'ort_guven': 0.8865159504132231, 'min_guven': 0.3353, 'max_rt': 22}
{'_id': 'Delta', 'toplam_tweet': 2222, 'ort_guven': 0.8698782628262827, 'min_guven': 0.3363, 'max_rt': 31}
{'_id': 'Virgin America', 'toplam_tweet': 504, 'ort_guven': 0.8760861111111111, 'min_guven': 0.3482, 'max_rt': 4}

Sorgu 5: Negatif tweetlerde havayolu bazli guven seviyesi etiketi ($cond):
{'_id': 'United', 'negatif_sayisi': 2633, 'ort_guven': 0.933, 'guven_seviyesi': 'yuksek'}
{'_id': 'US Airways', 'negatif_sayisi': 2263, 'ort_guven': 0.94

In [5]:
print("--- 3. Indeks, Update ve Delete Islemleri ---")

# sorgu 7: compound (bilesik) indeks olustur - airline + sentiment birlikte
# bu sayede ikisini birden filtreleyen sorgular cok hizlanir
print("\nSorgu 7: Bilesik indeks (airline + sentiment) olusturuluyor...")
collection.create_index([("airline", 1), ("airline_sentiment", 1)])

# indeksin gercekten kullanildigini explain ile dogrulayalim
aciklama = collection.find(
    {"airline": "Delta", "airline_sentiment": "negative"}
).explain()
kazanan_plan = aciklama['queryPlanner']['winningPlan']
# indeks bilgisi inputStage icinde olabilir
if 'inputStage' in kazanan_plan:
    print("Kullanilan indeks:", kazanan_plan['inputStage'].get('indexName', 'bulunamadi'))
else:
    print("Plan detayi:", kazanan_plan)

# sorgu 8: updateOne - $set ile birden fazla alan ekle + kosullu guncelleme
# en cok RT alan tweeti bul ve ozel olarak isaretle
print("\nSorgu 8: En cok RT alan tweete ozel etiket ($set - coklu alan):")
collection.update_one(
    {"retweet_count": {"$gt": 10}},
    {"$set": {
        "one_cikan": True,
        "kategori": "viral_tweet",
        "inceleme_notu": "yuksek etkilesim"
    }}
)
kontrol = collection.find_one({"one_cikan": True}, {"text": 1, "retweet_count": 1, "kategori": 1, "_id": 0})
print("Guncellenen kayit:", kontrol)

# sorgu 9: updateMany - $set + $inc birlikte kullanimi
# guven skoru 1.0 olan tum tweetlere "kesin_sonuc: true" ekle
# ayni zamanda retweet_count degerini 1 artir ($inc)
print("\nSorgu 9: Guven=1.0 olanlara alan ekle + RT sayisini 1 artir ($set + $inc):")
guncelleme = collection.update_many(
    {"airline_sentiment_confidence": 1.0},
    {
        "$set": {"kesin_sonuc": True},
        "$inc": {"retweet_count": 1}
    }
)
print(f"Guncellenen kayit sayisi: {guncelleme.modified_count}")

# sorgu 10: deleteMany - karmasik kosulla silme
# guven skoru 0.5 in altinda VE sebebi "Can't Tell" olan kayitlar analize zarar veriyor, silelim
print("\nSorgu 10: Dusuk guvenli + belirsiz sebepli kayitlari sil ($lt + $and):")
silinen = collection.delete_many({
    "$and": [
        {"airline_sentiment_confidence": {"$lt": 0.5}},
        {"negativereason": "Can't Tell"}
    ]
})
print(f"Silinen kayit sayisi: {silinen.deleted_count}")

--- 3. Indeks, Update ve Delete Islemleri ---

Sorgu 7: Bilesik indeks (airline + sentiment) olusturuluyor...
Kullanilan indeks: airline_1_airline_sentiment_1

Sorgu 8: En cok RT alan tweete ozel etiket ($set - coklu alan):
Guncellenen kayit: {'retweet_count': 22, 'text': '@SouthwestAir beautiful day in Seattle! http://t.co/iqu0PPVq2S', 'kategori': 'viral_tweet'}

Sorgu 9: Guven=1.0 olanlara alan ekle + RT sayisini 1 artir ($set + $inc):
Guncellenen kayit sayisi: 10409

Sorgu 10: Dusuk guvenli + belirsiz sebepli kayitlari sil ($lt + $and):
Silinen kayit sayisi: 25


In [6]:
print("--- 4. Ileri Seviye Sorgular ---")

# sorgu 11: gelismis regex - "cancel" VEYA "delay" iceren tweetleri bul
# iki kelimeyi tek regex ile yakaliyorum (pipe | operatoru)
# ayrica sadece United ve Delta icin filtrele ($in ile birlesik kullanim)
print("\nSorgu 11: 'cancel' veya 'delay' iceren United/Delta tweetleri (Regex + $in):")
sonuc11 = collection.find(
    {
        "text": {"$regex": "cancel|delay", "$options": "i"},
        "airline": {"$in": ["United", "Delta"]}
    },
    {"airline": 1, "text": 1, "negativereason": 1, "_id": 0}
).sort("airline_sentiment_confidence", -1).limit(3)
for doc in sonuc11:
    print(doc)

# sorgu 12: $in ile aggregation pipeline birlesimi
# sadece buyuk 3 havayolunu al, her birinin duygu dagilimini (pos/neg/neutral sayilarini) cikar
print("\nSorgu 12: Buyuk 3 havayolunun duygu dagilimi ($in + aggregation):")
pipeline_ileri = [
    {"$match": {"airline": {"$in": ["United", "American", "US Airways"]}}},
    {"$group": {
        "_id": {"havayolu": "$airline", "duygu": "$airline_sentiment"},
        "adet": {"$sum": 1}
    }},
    {"$sort": {"_id.havayolu": 1, "adet": -1}}
]
for doc in collection.aggregate(pipeline_ileri):
    print(doc)

--- 4. Ileri Seviye Sorgular ---

Sorgu 11: 'cancel' veya 'delay' iceren United/Delta tweetleri (Regex + $in):
{'negativereason': 'Cancelled Flight', 'airline': 'Delta', 'text': "@JetBlue but by Cancelled Flighting my flight and pushing me to the next day I'd lose $150 hotel which was why I was trying to get a same-day flight."}
{'negativereason': 'Cancelled Flight', 'airline': 'Delta', 'text': '@JetBlue Cancelled Flighted my flight. Went with another airline 2 leave 2day. They Cancelled Flighted also. Called JetBlue &amp; got same flight but now $250 moreð\x9f\x91º'}
{'negativereason': 'Late Flight', 'airline': 'Delta', 'text': '@JetBlue why was Flight 1856 delayed to Buffalo ?  Itâ\x80\x99s a direct flight and the plane is at the gate.'}

Sorgu 12: Buyuk 3 havayolunun duygu dagilimi ($in + aggregation):
{'_id': {'havayolu': 'American', 'duygu': 'negative'}, 'adet': 1941}
{'_id': {'havayolu': 'American', 'duygu': 'neutral'}, 'adet': 455}
{'_id': {'havayolu': 'American', 'duygu': 'posi